In [1]:
from collections.abc import Callable
from pathlib import Path
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import seaborn as sns
import sys
sys.path.insert(0, str(Path.cwd().parent))


from src.encoder import DummyEncoder
from src import quant, search

HOME_DIR = Path("/Users/arnoochka/Study/Uinversity/Lossy-RAG")



/Users/arnoochka/miniconda3/envs/lossy-rag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1111 18:59:26.408000 1424 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
data_dir = Path("../data")
chunks = pd.read_csv(data_dir / "chunks.csv")
questions = pd.read_csv(data_dir / "questions.csv")
answer_chunks = pd.read_csv(data_dir / "answer_chunks.csv")

chunks.head()

,chunk_id,pid,chunk_no,start,end,text
0,0,0,0,0,127,"Architecturally, the school has a Catholic cha..."
1,1,0,1,54,270,Atop the Main Building's gold dome is a golden...
2,2,0,2,128,333,Immediately in front of the Main Building and ...
3,3,0,3,271,421,Next to the Main Building is the Basilica of t...
4,4,0,4,334,550,"Immediately behind the basilica is the Grotto,..."


In [3]:
model_name = "BAAI/bge-small-en-v1.5"
save_dir = data_dir / f"embeddings_{model_name.rsplit('/', 1)[-1]}"

chunk_ids = np.load(save_dir / "chunk_ids.npy")
chunk_embeddings = np.load(save_dir / "chunk_embeddings.npy")
query_ids = np.load(save_dir / "query_ids.npy")
query_embeddings = np.load(save_dir / "query_embeddings.npy")

chunks = chunks.set_index("chunk_id", drop=False).loc[chunk_ids].reset_index(drop=True)
questions = questions.set_index("qid", drop=False).loc[query_ids].reset_index(drop=True)


In [4]:
encoder = DummyEncoder(
    chunk_texts=chunks["text"].tolist(),
    chunk_embeddings=chunk_embeddings,
    query_texts=questions["question"].tolist(),
    query_embeddings=query_embeddings,
)


In [5]:
def hit(retrieved: np.ndarray, relevant: set[int]) -> float:
    """Есть ли среди найденных фрагментов хотя бы один эталонный."""
    return float(len(set(retrieved) & relevant) > 0)


def precision(retrieved: np.ndarray, relevant: set[int]) -> float:
    """Доля эталонных фрагментов среди найденных."""
    retrieved_set = set(retrieved)
    return len(retrieved_set & relevant) / len(retrieved_set)


def recall(retrieved: np.ndarray, relevant: set[int]) -> float:
    """Доля найденных фрагментов среди всех эталонных."""
    return len(set(retrieved) & relevant) / len(relevant)


def mrr(retrieved: np.ndarray, relevant: set[int]) -> float:
    """Обратный ранг первого эталонного фрагмента; 0, если его нет."""
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in relevant:
            return 1.0 / rank
    return 0.0


def calculate_metrics(
    top_ids: np.ndarray,
    questions: pd.DataFrame, 
    answer_chunks: pd.DataFrame,
    metric_fn: Callable[[np.ndarray, set[int]], float],
    ks: tuple[int,...] = (1, 5, 10),
) -> pd.DataFrame:
    """Усредняет выбранную метрику по строгому и мягкому эталонам."""
    metrics = {}
    for name, gold in (
        ("strict", answer_chunks[answer_chunks["is_annotated"]]),
        ("soft", answer_chunks),
    ):
        gold_chunks = gold.groupby("orig_qid")["chunk_id"].agg(set).to_dict()
        metrics[name] = []
        for k in ks:
            values = [
                metric_fn(row[:k], gold_chunks[orig_qid])
                for orig_qid, row in zip(questions["orig_qid"], top_ids)
            ]
            metrics[name].append(np.mean(values))

    return pd.DataFrame(metrics, index=pd.Index(ks, name="k"))

In [6]:
def evaluate_search(
    chunks: pd.DataFrame,
    questions: pd.DataFrame,
    answer_chunks: pd.DataFrame,
    metric_fns: tuple[Callable[[np.ndarray, set[int]], float], ...],
    searcher: search.BaseSearch,
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, pd.DataFrame]:
    """Обучает поиск на train-текстах и считает метрики train/valid по одной выдаче.

    Поиск идёт по всей базе. Исходные вопросы и перефразы оцениваются отдельно.
    """
    train_pids = questions.loc[questions["role"] == "train", "pid"]
    chunk_train_mask = chunks["pid"].isin(train_pids)
    searcher.fit(chunks.loc[chunk_train_mask, "text"].tolist())
    searcher.add(chunks["text"].tolist(), chunks["chunk_id"].to_numpy())
    top_ids, _ = searcher.search(questions["question"].tolist(), k=max(ks))
    metrics = {}
    for metric_fn in metric_fns:
        results = []
        for role in ("train", "valid"):
            for question_type, is_paraphrase in (("real", False), ("paraphrase", True)):
                mask = (
                    (questions["role"] == role)
                    & (questions["is_paraphrase"] == is_paraphrase)
                ).to_numpy()
                group_metrics = calculate_metrics(
                    top_ids[mask], questions.loc[mask], answer_chunks, metric_fn, ks,
                )
                group_metrics.columns = [
                    f"{role}_{name}_{question_type}" for name in group_metrics.columns
                ]
                results.append(group_metrics)
        metrics[metric_fn.__name__] = pd.concat(results, axis=1)
    return metrics


In [7]:
from tqdm import tqdm

def conduct_experiments(exps: dict[str, search.BaseSearch],
                        save_dir: Path,
                        metric_fns: tuple[Callable[[np.ndarray, set[int]], float], ...],
                        ks: tuple[int,...]) -> None:

    for name, searcher in tqdm(exps.items()):
        results = evaluate_search(
            chunks=chunks,
            questions=questions,
            answer_chunks=answer_chunks,
            metric_fns=metric_fns,
            searcher=searcher,
            ks=ks)
        for metric_name, metrics in results.items():
            metric_dir = save_dir / metric_name
            metric_dir.mkdir(parents=True, exist_ok=True)
            save_path = metric_dir / f"exp_{name}.csv"
            metrics.to_csv(save_path, index=False)
            print(f"{name}: {save_path}")

KS = (1, 5, 10, 15, 20, 25)


In [8]:
experiments = {
    "base": search.BaseSearch(encoder=encoder, quantizer=quant.BaseQuant()),
}
conduct_experiments(exps=experiments,
                    save_dir=HOME_DIR / "results/search",
                    metric_fns=(hit, mrr),
                    ks=KS)


100%|██████████| 1/1 [01:08<00:00, 68.25s/it]

base: /Users/arnoochka/Study/Uinversity/Lossy-RAG/results/search/hit/exp_base.csv
base: /Users/arnoochka/Study/Uinversity/Lossy-RAG/results/search/mrr/exp_base.csv


In [ ]:
experiments = {
    f"ivf_{nprobe}": search.IvfSearch(
        encoder=encoder,
        quantizer=quant.BaseQuant(),
        nlist=1024,
        nprobe=nprobe,
    )
    for nprobe in (4, 16, 64)
}
conduct_experiments(exps=experiments,
                    save_dir=HOME_DIR / "results/search",
                    metric_fns=(hit, mrr),
                    ks=KS)


 33%|███▎      | 1/3 [00:22<00:45, 22.98s/it]

ivf_4: /Users/arnoochka/Study/Uinversity/Lossy-RAG/results/search/hit/exp_ivf_4.csv
ivf_4: /Users/arnoochka/Study/Uinversity/Lossy-RAG/results/search/mrr/exp_ivf_4.csv


 67%|██████▋   | 2/3 [01:13<00:39, 39.12s/it]

ivf_16: /Users/arnoochka/Study/Uinversity/Lossy-RAG/results/search/hit/exp_ivf_16.csv
ivf_16: /Users/arnoochka/Study/Uinversity/Lossy-RAG/results/search/mrr/exp_ivf_16.csv


In [ ]:
experiments = {
    f"hnsw_{ef_search}": search.HnswSearch(
        encoder=encoder,
        quantizer=quant.BaseQuant(),
        M=16,
        ef_construction=200,
        ef_search=ef_search,
    )
    for ef_search in (25, 100, 400)
}
conduct_experiments(exps=experiments,
                    save_dir=HOME_DIR / "results/search",
                    metric_fns=(hit, mrr),
                    ks=KS)


In [ ]:
experiments = {
    f"bm25_{b}": search.Bm25Search(
        encoder=encoder,
        quantizer=quant.BaseQuant(),
        k1=1.2,
        b=b,
    )
    for b in (0, 0.75)
}
conduct_experiments(exps=experiments,
                    save_dir=HOME_DIR / "results/search",
                    metric_fns=(hit, mrr),
                    ks=KS)


In [ ]:
experiments = {
    f"ivf_{nprobe}": search.IvfRaBitqSearch(
        encoder=encoder,
        quantizer=quant.BaseQuant(),
        nlist=1024,
        nprobe=nprobe,
        epsilon=1.9,
    )
    for nprobe in (4, 16, 64)
}
conduct_experiments(exps=experiments,
                    save_dir=HOME_DIR / "results/search",
                    metric_fns=(hit, mrr),
                    ks=KS)


In [ ]:
experiments = {
    f"hybrid_{b}_{alpha}": search.HybridBm25Search(
        vector_search=search.IvfSearch(
            encoder=encoder,
            quantizer=quant.BaseQuant(),
            nlist=1024,
            nprobe=16,
        ),
        alpha=alpha,
        num_candidates=100,
        k1=1.2,
        b=b,
    )
    for b in (0, 0.75)
    for alpha in (0.25, 0.5, 0.75)
}
conduct_experiments(exps=experiments,
                    save_dir=HOME_DIR / "results/search",
                    metric_fns=(hit, mrr),
                    ks=KS)
